# Group 8 — Bandit Analysis Verification

**Paper:** Elise Wolf (2025), *A Framework for Fair Evaluation of Variance-Aware Bandit Algorithms*, arXiv:2510.27001

**Author's codebase:** github.com/eelisee/bandit_playground (branch `v1.0-SKILL2025`)

This notebook verifies every number used in the Group 8 presentation. It checks:

1. Values transcribed from the paper's Tables 3–5 against the author's **published CSV output**
2. Derived figures we computed ourselves (percentage reductions, variance differences)
3. The regret identity: regret ≈ gap × suboptimal pulls

**Scope:** this does *not* rerun the paper's 100-trial, one-million-step simulation.

In [ ]:
import math

HORIZON = 1_000_000

# Transcribed from the paper's Tables 3-5.
# Suboptimal ratios are taken from the author's published CSV output
# (column 'Average Suboptimal Arms' at the T = 1,000,000 checkpoint).
scenarios = {
    "A": {"p_sub": 0.800, "p_opt": 0.900, "ucb": 238.17, "ucb_tuned": 30.67,
          "etc": 10.10, "tuned_sub_ratio": 0.00030669},
    "B": {"p_sub": 0.895, "p_opt": 0.900, "ucb": 1127.17, "ucb_tuned": 212.21,
          "etc": 150.66, "tuned_sub_ratio": 0.04244215},
    "C": {"p_sub": 0.890, "p_opt": 0.895, "ucb": 1172.57, "ucb_tuned": 226.09,
          "etc": 170.53, "tuned_sub_ratio": 0.04521790},
}

def gap(s):                 return s["p_opt"] - s["p_sub"]
def bernoulli_variance(p):  return p * (1 - p)
def regret_from_ratio(s):   return gap(s) * s["tuned_sub_ratio"] * HORIZON
def pct_reduction(base, c): return (base - c) / base * 100

print("Setup loaded for Scenarios A, B and C.")

## Cross-check against the author's published simulation output

Each row below pairs a value printed in the paper with the value in the author's own CSV file.
File names encode arm probabilities: `800_900` = Scenario A, `895_900` = B, `890_895` = C.

In [ ]:
# (algorithm, scenario, paper value, repo file under data/, repo value)
repo_crosscheck = [
    ("ETC (m=100)",    "A",    10.10, "ETC/exploration_rounds_100/average_results_800_900.csv",     10.0950),
    ("UCB-Tuned",      "A",    30.67, "UCB-Tuned/default/average_results_800_900.csv",              30.6690),
    ("EUCBV",          "A",    47.70, "EUCBV/rho_0_5/average_results_800_900.csv",                  47.7270),
    ("UCB",            "A",   238.17, "UCB/default/average_results_800_900.csv",                   238.1750),
    ("UCB-Improved",   "A", 32500.11, "UCB-Improved/delta_1/average_results_800_900.csv",        32500.1060),
    ("ETC (m=10000)",  "B",   150.66, "ETC/exploration_rounds_10000/average_results_895_900.csv",  150.6631),
    ("UCB-Tuned",      "B",   212.21, "UCB-Tuned/default/average_results_895_900.csv",             212.2084),
    ("UCB",            "B",  1127.17, "UCB/default/average_results_895_900.csv",                  1127.1738),
    ("ETC (m=10000)",  "C",   170.53, "ETC/exploration_rounds_10000/average_results_890_895.csv",  170.5346),
    ("UCB-Tuned",      "C",   226.09, "UCB-Tuned/default/average_results_890_895.csv",             226.0870),
    ("UCB",            "C",  1172.57, "UCB/default/average_results_890_895.csv",                  1172.5742),
]

print('{:<16}{:<4}{:>12}{:>14}   Match'.format('Algorithm','Sc','Paper','Author CSV'))
print("-" * 60)
for algo, sc, paper, path, repo in repo_crosscheck:
    ok = math.isclose(paper, repo, rel_tol=1e-3)
    assert ok, f"MISMATCH: {algo} {sc}"
    print('{:<16}{:<4}{:>12,.2f}{:>14,.4f}   {}'.format(algo, sc, paper, repo, 'OK' if ok else 'FAIL'))

print(f"\n{len(repo_crosscheck)} of {len(repo_crosscheck)} values match the author's published output.")

## Contributor checks

- **Chandan** — scenario gaps, difficulty ratio, regret identity
- **Divakaran** — Bernoulli variances and the variance clip in `UCB_Tuned.py`
- **Madhulatha** — percentage reductions from Tables 3–5
- **Akash** — variance coupling and the §4.3 baseline variance figure

In [ ]:
# Chandan - difficulty and the regret identity
difficulty_ratio = gap(scenarios["A"]) / gap(scenarios["B"])
regret_b = regret_from_ratio(scenarios["B"])
regret_c = regret_from_ratio(scenarios["C"])

# Divakaran - Bernoulli variances; the UCB-Tuned clip is min(1/4, V_ks)
variances = {p: bernoulli_variance(p) for p in (0.800, 0.890, 0.895, 0.900)}
max_variance = max(variances.values())

# Madhulatha - percentage reductions
reduction_b = pct_reduction(scenarios["B"]["ucb"], scenarios["B"]["ucb_tuned"])
reduction_c = pct_reduction(scenarios["C"]["ucb"], scenarios["C"]["ucb_tuned"])
etc_adv_b   = pct_reduction(scenarios["B"]["ucb_tuned"], scenarios["B"]["etc"])
etc_adv_c   = pct_reduction(scenarios["C"]["ucb_tuned"], scenarios["C"]["etc"])

# Akash - variance coupling, and the paper's own baseline variance figure
var_diff_b = bernoulli_variance(0.895) - bernoulli_variance(0.900)
var_diff_c = bernoulli_variance(0.890) - bernoulli_variance(0.895)
paper_baseline_variance = HORIZON * bernoulli_variance(0.895)

assert math.isclose(difficulty_ratio, 20.0, abs_tol=1e-9)
assert math.isclose(regret_b, 212.21, abs_tol=0.01)
assert math.isclose(regret_c, 226.09, abs_tol=0.01)
assert math.isclose(max_variance, 0.16, abs_tol=1e-12)
assert math.isclose(reduction_b, 81.1732036871102, abs_tol=1e-9)
assert math.isclose(reduction_c, 80.71842192790196, abs_tol=1e-9)
assert math.isclose(var_diff_b, 0.003975, abs_tol=1e-12)
assert math.isclose(var_diff_c, 0.003925, abs_tol=1e-12)
assert math.isclose(paper_baseline_variance, 93_975.0, abs_tol=1e-6)

print("All contributor calculations executed successfully.")

## Final summary

Run this cell last. Screenshot it together with the notebook name and the Colab runtime indicator.

In [ ]:
print("GROUP 8 - EXECUTABLE VERIFICATION SUMMARY")
print("=" * 64)
print(f"Repo cross-check  | paper vs author CSV        | {len(repo_crosscheck)}/{len(repo_crosscheck)} values match")
print(f"Chandan           | micro-gap difficulty ratio | {difficulty_ratio:.2f}x")
print(f"Chandan           | regret identity B / C      | {regret_b:.2f} / {regret_c:.2f}")
print(f"Divakaran         | max Bernoulli variance     | {max_variance:.4f} (clip at 0.25 never binds)")
print(f"Madhulatha        | UCB-Tuned reduction B / C  | {reduction_b:.2f}% / {reduction_c:.2f}%")
print(f"Madhulatha        | tuned ETC advantage B / C  | {etc_adv_b:.2f}% / {etc_adv_c:.2f}%")
print(f"Akash             | variance difference B / C  | {var_diff_b:.6f} / {var_diff_c:.6f}")
print(f"Akash             | baseline variance, p=.895  | {paper_baseline_variance:,.0f} (paper Sec. 4.3: 93,975)")
print("=" * 64)
print("ALL CHECKS PASSED")
print("Scope: verified against paper Tables 3-5 and the author's published")
print("CSV output. Not a rerun of the full 100-trial simulation.")

## How to run this

1. Open in Google Colab.
2. **Runtime → Run all**.
3. Confirm the last cell prints **ALL CHECKS PASSED**.
4. Screenshot the final cell with the notebook name and runtime indicator visible.

Share the notebook as **Anyone with the link — Viewer**, never as editor.